## PyINE code editing demo

This notebook shows how to edit code snippets with LLMs using multiple prompts that are part of the PyINE framework.

This kind of code editing process is a core requirement for the creation of our evaluation scenarios.

In [ ]:
import importlib

import pyine.prompts.manager
import pyine.utils.code.patching
import pyine.utils.llm_providers
import pyine.utils.portability
import pyine.utils.reprod

pyine.utils.reprod.load_dotenv()

importlib.reload(pyine.prompts.manager)
importlib.reload(pyine.utils.code.patching)
importlib.reload(pyine.utils.llm_providers)
importlib.reload(pyine.utils.portability)

In [ ]:
import langchain_core.globals
import langchain_core.messages

langchain_core.globals.set_debug(True)

In [ ]:
example_snippet = """\
from itertools import tee, islice
from collections import deque

def read_numbers():
    '''Lazy reader that yields ints from stdin until an empty line.'''
    while (line := input().strip()):
        yield from (int(x) for x in line.split())

def chunk_increasing(seq):
    '''Yield lists of consecutive, strictly-increasing numbers.'''
    buf = []
    for n in seq:
        if not buf or n > buf[-1]:
            buf.append(n)
        else:              # break in monotonicity
            yield buf
            buf = [n]
    if buf:
        yield buf

def analyze(chunks):
    '''Return longest increasing chunk and its cumulative rolling sums.'''
    longest = max(chunks, key=len, default=[])
    # rolling sums using deque as a sliding window
    win, acc = deque(), []
    running = 0
    for n in longest:
        win.append(n)
        running += n
        acc.append(running)
    return longest, acc

def pretty_print(longest, sums):
    '''Display results.'''
    print("Longest increasing subsequence:")
    print(" ", longest)
    print("Cumulative sums:")
    print(" ", sums)

def main():
    '''Glue everything together.'''
    nums = read_numbers()               # generator
    a, b = tee(chunk_increasing(nums))  # reuse chunks twice
    longest, sums = analyze(a)          # first consumption
    pretty_print(longest, sums)
    # show summary statistics (requires second tee’ed iterator)
    lengths = [len(c) for c in b]
    if lengths:
        print("Average chunk length:", sum(lengths) / len(lengths))

if __name__ == "__main__":
    main()
"""
description = None
example_inputs = """\
3 5 7 1 2
2 3 4
0
"""
example_expected_output = """\
Longest increasing subsequence:
  [3, 5, 7]
Cumulative sums:
  [3, 8, 15]
Average chunk length: 2.25
"""

print("Example snippet:")
pyine.utils.portability.print_code_with_numbered_lines(
    example_snippet,
)
print(f"\nExample inputs: {example_inputs}")
print(f"Example expected output: {example_expected_output}")
invocation_kwargs = dict(
    code=example_snippet,
    description=description,
    inputs=example_inputs,
    expected_exec_output=example_expected_output,
)

In [ ]:
available_prompts = pyine.prompts.manager.get_framework_prompt_manager().list_prompts()
available_prompts_str = ", ".join(available_prompts)
print(f"Available prompt names: {available_prompts_str}")

In [ ]:
target_prompt_name = "issues/iterators"  # CHANGE ME FOR DIFFERENT CODE EDITING DEMOS (see available prompts above)

print(f"Will prepare and use '{target_prompt_name}' prompt...")
assert target_prompt_name in available_prompts, f"invalid prompt, please choose from: {available_prompts_str}"
prompt_config = pyine.prompts.manager.get_prompt_config(target_prompt_name)
print(f"Prompt description:\n{prompt_config.metadata.description}\n")
prompt_template = pyine.prompts.manager.get_prompt_template(target_prompt_name)
prompt = prompt_template.format(**invocation_kwargs)
print(f"Prompt '{target_prompt_name}':\n{prompt}")
max_tokens = 10_000  # if too little, we might not get any response at all (all be consumed reasoning)
openai_o3 = pyine.utils.llm_providers.get_model_from_provider(
    provider="openai",
    model="o3",
    temperature=1.0,  # only value supported by o3
    max_tokens=max_tokens,
)
# NOTE: depending on the target prompt, might want to use pass in pydantic model for structured output
prompt_chain = pyine.prompts.manager.get_prompt_chain(model=openai_o3, prompt_name=target_prompt_name)
response = prompt_chain.invoke(invocation_kwargs)
assert isinstance(response, langchain_core.messages.base.BaseMessage)
print(f"\n\nFinal prediction:\n{response.content}\n\n")
input_tokens = response.usage_metadata.get("input_tokens", None)
output_tokens = response.usage_metadata.get("output_tokens", None)
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
if (output_tokens or 0) > max_tokens * 0.8:
    print("Output token usage exceeds 80% of max tokens, adjust max token limit.")

In [ ]:
diff = pyine.utils.code.patching.compute_patch(
    original=example_snippet,
    modified=response.content,
)
print(f"Diff for '{target_prompt_name}' prompt results:")
pyine.utils.code.patching.show_colored_diff(diff)